### Imports + robust repo root + paths (no more wrong Day-19/Day-11 paths)

In [1]:
import os
import json
import uuid
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import duckdb


In [2]:
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "Day-11").exists() and (p / ".git").exists():
            return p
    # fallback: first parent that has Day-11
    for p in [start] + list(start.parents):
        if (p / "Day-11").exists():
            return p
    return start

REPO_ROOT = find_repo_root(Path.cwd())
DAY19_DIR = REPO_ROOT / "Day-19"
REPORTS  = DAY19_DIR / "reports"
ARTIFACTS = DAY19_DIR / "artifacts"
SRC = DAY19_DIR / "src"

for d in [REPORTS, ARTIFACTS, SRC]:
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = REPO_ROOT / "Day-11" / "data" / "warehouse" / "day11_noshow.duckdb"

if not DB_PATH.exists():
    hits = list(REPO_ROOT.glob("**/*.duckdb"))
    if not hits:
        raise FileNotFoundError(f"Could not find any .duckdb under: {REPO_ROOT}")
    DB_PATH = hits[0]

print("REPO_ROOT:", REPO_ROOT)
print("DB_PATH   :", DB_PATH)
print("DAY19_DIR :", DAY19_DIR)


REPO_ROOT: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science
DB_PATH   : C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-11\data\warehouse\day11_noshow.duckdb
DAY19_DIR : C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-19


### Open DuckDB (read/write) + confirm tables

In [3]:
# If you get "file is being used by another process":
# close other notebooks using DuckDB, restart kernel, rerun from Cell 1.

con = duckdb.connect(str(DB_PATH))  # read/write
tables = con.execute("SHOW TABLES").df()
print(tables)


                                          name
0                          bronze_appointments
1                       gold_appointments_base
2                gold_appointments_features_v1
3  gold_appointments_features_v1_patient_split
4                     gold_appointments_splits
5                          silver_appointments
6                             split_patient_v1


### Set table names and ensure splits table exists

In [4]:
FEATURES_TBL = "gold_appointments_features_v1"
SPLITS_TBL   = "gold_appointments_splits"

existing = set(con.execute("SHOW TABLES").df()["name"].tolist())

if FEATURES_TBL not in existing:
    raise ValueError(f"Missing {FEATURES_TBL}. Available tables: {sorted(existing)[:20]}")

if SPLITS_TBL not in existing:
    print("Splits table not found. Creating:", SPLITS_TBL)

    people = con.execute(f"""
        SELECT DISTINCT person_id
        FROM {FEATURES_TBL}
        WHERE person_id IS NOT NULL
    """).df()

    people = np.sort(people["person_id"].unique())
    rng = np.random.default_rng(42)
    rng.shuffle(people)

    n = len(people)
    n_train = int(round(0.70 * n))
    n_valid = int(round(0.15 * n))
    train_ids = people[:n_train]
    valid_ids = people[n_train:n_train+n_valid]
    test_ids  = people[n_train+n_valid:]

    splits = pd.DataFrame({"person_id": people})
    splits["split"] = "test"
    splits.loc[splits["person_id"].isin(train_ids), "split"] = "train"
    splits.loc[splits["person_id"].isin(valid_ids), "split"] = "valid"

    con.register("splits_df", splits)
    con.execute(f"CREATE OR REPLACE TABLE {SPLITS_TBL} AS SELECT * FROM splits_df")
    print("Wrote:", SPLITS_TBL)

print("OK — FEATURES_TBL:", FEATURES_TBL)
print("OK — SPLITS_TBL  :", SPLITS_TBL)


OK — FEATURES_TBL: gold_appointments_features_v1
OK — SPLITS_TBL  : gold_appointments_splits


### Pull modeling frame (features + split)

In [5]:
df = con.execute(f"""
    SELECT f.*, s.split
    FROM {FEATURES_TBL} f
    JOIN {SPLITS_TBL} s
      ON f.person_id = s.person_id
""").df()

A_COL = "sms_received"
Y_COL = "label"

df = df.dropna(subset=[A_COL, Y_COL, "split"]).copy()
df[A_COL] = df[A_COL].astype(int)
df[Y_COL] = df[Y_COL].astype(int)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nA counts:", df[A_COL].value_counts().to_dict())
print("Y counts:", df[Y_COL].value_counts().to_dict())
print("\nNaive mean(Y) by A:\n", df.groupby(A_COL)[Y_COL].agg(mean_Y="mean", n="size"))


Shape: (391276, 26)
Columns: ['appointment_id', 'person_id', 'label', 'sms_received', 'age', 'gender', 'neighbourhood', 'scholarship', 'hipertension', 'diabetes', 'alcoholism', 'handcap', 'lead_time_days', 'lead_time_clipped', 'lead_time_log1p', 'lead_time_bin', 'appt_date', 'sched_date', 'appt_dow', 'appt_month', 'sched_dow', 'sched_month', 'sched_hour', 'nbhd_n', 'prior_appt_count', 'split']

A counts: {0: 290989, 1: 100287}
Y counts: {0: 321373, 1: 69903}

Naive mean(Y) by A:
                 mean_Y       n
sms_received                  
0             0.145311  290989
1             0.275400  100287


### Define covariates + preprocessing (safe OHE + scaling)

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

use_covars = [
    "age","gender","neighbourhood",
    "scholarship","hipertension","diabetes","alcoholism","handcap",
    "lead_time_days","lead_time_clipped","lead_time_log1p","lead_time_bin",
    "appt_dow","appt_month","sched_dow","sched_month","sched_hour",
    "nbhd_n","prior_appt_count"
]

X_cols = [c for c in use_covars if c in df.columns]

CAT_COLS = [c for c in ["gender","neighbourhood","lead_time_bin","appt_dow","appt_month","sched_dow","sched_month","sched_hour"] if c in X_cols]
NUM_COLS = [c for c in X_cols if c not in CAT_COLS]

print("X_cols:", len(X_cols))
print("Categorical:", CAT_COLS)
print("Numeric:", [c for c in NUM_COLS][:20])


X_cols: 19
Categorical: ['gender', 'neighbourhood', 'lead_time_bin', 'appt_dow', 'appt_month', 'sched_dow', 'sched_month', 'sched_hour']
Numeric: ['age', 'scholarship', 'hipertension', 'diabetes', 'alcoholism', 'handcap', 'lead_time_days', 'lead_time_clipped', 'lead_time_log1p', 'nbhd_n', 'prior_appt_count']


In [7]:
def make_preprocess(cat_cols, num_cols):
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler(with_mean=False))
            ]), num_cols),
            ("cat", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ohe", make_ohe())
            ]), cat_cols),
        ],
        remainder="drop"
    )


### Split masks

In [8]:
m_tr = df["split"].eq("train").to_numpy()
m_va = df["split"].eq("valid").to_numpy()
m_te = df["split"].eq("test").to_numpy()

print(df["split"].value_counts())


split
train    276859
test      57462
valid     56955
Name: count, dtype: int64


### Fit propensity model (for governance/positivity checks) + save pscore

In [9]:
X = df[X_cols].copy()
A = df[A_COL].to_numpy()

prep_ps = make_preprocess(CAT_COLS, NUM_COLS)
ps_model = LogisticRegression(max_iter=5000, solver="saga")

ps_pipe = Pipeline(steps=[("prep", prep_ps), ("clf", ps_model)])
ps_pipe.fit(X[m_tr], A[m_tr])

ps_valid = ps_pipe.predict_proba(X[m_va])[:, 1]
print("Propensity ROC-AUC (train->valid):", round(roc_auc_score(A[m_va], ps_valid), 4))

df["pscore"] = ps_pipe.predict_proba(X)[:, 1]
df[["pscore", A_COL, Y_COL]].head()


Propensity ROC-AUC (train->valid): 0.9227


,pscore,sms_received,label
0,0.000018,0,0
1,0.000054,0,0
2,0.000073,0,0
3,0.787422,0,1
4,0.783343,0,1


### Fit outcome models (mu1, mu0) safely (no feature-mismatch bugs)

In [10]:
Y = df[Y_COL].to_numpy()

# NEW preprocess objects for each pipeline (prevents the feature-count mismatch you hit)
prep_y1 = make_preprocess(CAT_COLS, NUM_COLS)
prep_y0 = make_preprocess(CAT_COLS, NUM_COLS)

y1_pipe = Pipeline(steps=[("prep", prep_y1), ("clf", LogisticRegression(max_iter=7000, solver="saga"))])
y0_pipe = Pipeline(steps=[("prep", prep_y0), ("clf", LogisticRegression(max_iter=7000, solver="saga"))])

y1_pipe.fit(X[m_tr & (A==1)], Y[m_tr & (A==1)])
y0_pipe.fit(X[m_tr & (A==0)], Y[m_tr & (A==0)])

# Potential outcomes on TEST
mu1_te = y1_pipe.predict_proba(X[m_te])[:, 1]   # P(no-show | do SMS)
mu0_te = y0_pipe.predict_proba(X[m_te])[:, 1]   # P(no-show | do no SMS)

uplift_te = mu0_te - mu1_te  # + means SMS helps (reduces no-show)

print("Mean uplift on TEST (mu0-mu1):", float(np.mean(uplift_te)))
print("Min/median/max uplift:", float(np.min(uplift_te)), float(np.median(uplift_te)), float(np.max(uplift_te)))


Mean uplift on TEST (mu0-mu1): -0.07051677157119358
Min/median/max uplift: -0.49823253985653465 -0.06189315900870618 0.5390283492916865


### Decision policies under budgets (expected policy value)

In [11]:
df_te = df.loc[m_te, ["appointment_id","person_id",A_COL,Y_COL,"split"] + X_cols].copy()
df_te["mu0"] = mu0_te
df_te["mu1"] = mu1_te
df_te["uplift"] = uplift_te

# scores for ranking
df_te["risk_score"] = df_te["mu0"]         # baseline no-SMS no-show risk
df_te["uplift_score"] = df_te["uplift"]    # expected reduction if SMS

def expected_no_show_rate_under_policy(df_in, k, strategy="risk"):
    n = len(df_in)
    k = int(min(max(k, 0), n))
    if k == 0:
        return float(df_in["mu0"].mean())

    if strategy == "risk":
        chosen = df_in.nlargest(k, "risk_score").index
    elif strategy == "uplift":
        chosen = df_in.nlargest(k, "uplift_score").index
    else:
        raise ValueError("strategy must be 'risk' or 'uplift'")

    exp_y = df_in["mu0"].copy()
    exp_y.loc[chosen] = df_in.loc[chosen, "mu1"]
    return float(exp_y.mean())

budgets = [0.01, 0.05, 0.10, 0.20]
rows = []
n_test = len(df_te)

none_rate = float(df_te["mu0"].mean())
all_rate  = float(df_te["mu1"].mean())

for frac in budgets:
    k = int(round(frac * n_test))
    risk_rate = expected_no_show_rate_under_policy(df_te, k, "risk")
    uplift_rate = expected_no_show_rate_under_policy(df_te, k, "uplift")
    rows.append({
        "budget_frac": frac,
        "k": k,
        "no_show_rate_none": none_rate,
        "no_show_rate_all": all_rate,
        "no_show_rate_risk_policy": risk_rate,
        "no_show_rate_uplift_policy": uplift_rate,
        "reduction_vs_none_risk": none_rate - risk_rate,
        "reduction_vs_none_uplift": none_rate - uplift_rate
    })

policy_table = pd.DataFrame(rows)
policy_table.to_csv(REPORTS / "DAY19_policy_value_table.csv", index=False)
policy_table


,budget_frac,k,no_show_rate_none,no_show_rate_all,no_show_rate_risk_policy,no_show_rate_uplift_policy,reduction_vs_none_risk,reduction_vs_none_uplift
0,0.01,575,0.204552,0.275069,0.201801,0.201243,0.002751,0.003309
1,0.05,2873,0.204552,0.275069,0.194447,0.192510,0.010105,0.012042
2,0.10,5746,0.204552,0.275069,0.188397,0.184947,0.016155,0.019605
3,0.20,11492,0.204552,0.275069,0.180180,0.175602,0.024372,0.028950


### Build “deployment outputs” for a chosen daily budget K

In [12]:
BUDGET_FRAC = 0.10
K = int(round(BUDGET_FRAC * len(df_te)))

df_te = df_te.sort_values("risk_score", ascending=False).copy()
df_te["rank_risk"] = np.arange(1, len(df_te)+1)

df_te2 = df_te.sort_values("uplift_score", ascending=False).copy()
df_te2["rank_uplift"] = np.arange(1, len(df_te2)+1)

# merge ranks back
df_dec = df_te.merge(df_te2[["appointment_id","rank_uplift"]], on="appointment_id", how="left")

# action flags
df_dec["send_sms_risk"] = (df_dec["rank_risk"] <= K).astype(int)
df_dec["send_sms_uplift"] = (df_dec["rank_uplift"] <= K).astype(int)

out_cols = [
    "appointment_id","person_id","split",
    "mu0","mu1","uplift",
    "rank_risk","rank_uplift",
    "send_sms_risk","send_sms_uplift"
]
df_dec[out_cols].to_csv(REPORTS / f"DAY19_decisions_test_k{K}.csv", index=False)

# target lists (top K)
df_dec.loc[df_dec["send_sms_risk"].eq(1), out_cols].head(K).to_csv(REPORTS / f"DAY19_target_list_risk_k{K}.csv", index=False)
df_dec.loc[df_dec["send_sms_uplift"].eq(1), out_cols].head(K).to_csv(REPORTS / f"DAY19_target_list_uplift_k{K}.csv", index=False)

print("Saved decisions + target lists in:", REPORTS)
print("K:", K, "out of", len(df_dec))


Saved decisions + target lists in: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-19\reports
K: 5746 out of 691834


### Create an input template CSV (what ops would send you daily)

In [13]:
template_cols = ["appointment_id","person_id"] + X_cols
template = df_te[template_cols].head(1).copy()
template.to_csv(ARTIFACTS / "DAY19_input_template.csv", index=False)

print("Wrote:", ARTIFACTS / "DAY19_input_template.csv")
template


Wrote: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-19\artifacts\DAY19_input_template.csv


,appointment_id,person_id,age,gender,neighbourhood,scholarship,hipertension,diabetes,alcoholism,handcap,...,lead_time_clipped,lead_time_log1p,lead_time_bin,appt_dow,appt_month,sched_dow,sched_month,sched_hour,nbhd_n,prior_appt_count
189836,5586010,66566534618649,15.0,M,ROMÃO,0,0,0,0,3,...,18,2.944439,15_30,2,5,5,4,3,2214,0


### Write policy outputs back into DuckDB (runs + recommendations)

In [15]:
RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.now().isoformat(timespec="seconds")

runs_df = pd.DataFrame([{
    "run_id": RUN_ID,
    "run_ts": RUN_TS,
    "features_table": FEATURES_TBL,
    "splits_table": SPLITS_TBL,
    "budget_k": int(K),
    "budget_frac": float(BUDGET_FRAC),
    "notes": "Day 19 decision outputs: risk + uplift policies from mu0/mu1"
}])

con.execute("""
CREATE TABLE IF NOT EXISTS policy_runs (
    run_id VARCHAR,
    run_ts VARCHAR,
    features_table VARCHAR,
    splits_table VARCHAR,
    budget_k BIGINT,
    budget_frac DOUBLE,
    notes VARCHAR
)
""")

con.register("runs_df", runs_df)
con.execute("INSERT INTO policy_runs SELECT * FROM runs_df")

# recommendations table: store BOTH strategies for same run_id
rec_risk = df_dec[out_cols].copy()
rec_risk["run_id"] = RUN_ID
rec_risk["strategy"] = "risk"
rec_risk["budget_k"] = int(K)
rec_risk["send_sms"] = rec_risk["send_sms_risk"]
rec_risk["rank"] = rec_risk["rank_risk"]

rec_uplift = df_dec[out_cols].copy()
rec_uplift["run_id"] = RUN_ID
rec_uplift["strategy"] = "uplift"
rec_uplift["budget_k"] = int(K)
rec_uplift["send_sms"] = rec_uplift["send_sms_uplift"]
rec_uplift["rank"] = rec_uplift["rank_uplift"]

rec_all = pd.concat([rec_risk, rec_uplift], ignore_index=True)

rec_all = rec_all[[
    "run_id","strategy","budget_k","appointment_id","person_id","split",
    "mu0","mu1","uplift","send_sms","rank"
]].copy()

con.execute("""
CREATE TABLE IF NOT EXISTS policy_recommendations (
    run_id VARCHAR,
    strategy VARCHAR,
    budget_k BIGINT,
    appointment_id BIGINT,
    person_id BIGINT,
    split VARCHAR,
    mu0 DOUBLE,
    mu1 DOUBLE,
    uplift DOUBLE,
    send_sms INTEGER,
    rank BIGINT
)
""")

con.register("rec_all_df", rec_all)
con.execute("INSERT INTO policy_recommendations SELECT * FROM rec_all_df")

print("Wrote policy_recommendations. Rows inserted:", len(rec_all))
print("RUN_ID:", RUN_ID)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote policy_recommendations. Rows inserted: 1383668
RUN_ID: b88cb6d2-2719-4277-870a-9f5445546589


### Save models to artifacts (for Day 20 packaging)

In [16]:
import joblib

joblib.dump(ps_pipe, ARTIFACTS / "day19_propensity_pipe.joblib")
joblib.dump(y0_pipe, ARTIFACTS / "day19_outcome_mu0_pipe.joblib")
joblib.dump(y1_pipe, ARTIFACTS / "day19_outcome_mu1_pipe.joblib")

meta = {
    "run_id": RUN_ID,
    "run_ts": RUN_TS,
    "X_cols": X_cols,
    "CAT_COLS": CAT_COLS,
    "NUM_COLS": NUM_COLS,
    "A_COL": A_COL,
    "Y_COL": Y_COL,
    "budget_k": int(K),
    "budget_frac": float(BUDGET_FRAC),
    "features_table": FEATURES_TBL,
    "splits_table": SPLITS_TBL,
}
with open(ARTIFACTS / "DAY19_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved artifacts in:", ARTIFACTS)


Saved artifacts in: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-19\artifacts


### Create Day-19/src/run_policy.py (batch scoring script)

In [17]:
run_policy_py = r'''
import argparse
import json
from pathlib import Path
import pandas as pd
import joblib

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True, help="CSV with appointment_id, person_id, and feature columns")
    ap.add_argument("--output", required=True, help="Output CSV path")
    ap.add_argument("--budget_k", type=int, required=True, help="Daily SMS budget K")
    ap.add_argument("--strategy", choices=["risk","uplift"], default="uplift")
    ap.add_argument("--artifacts_dir", default=str(Path(__file__).resolve().parents[1] / "artifacts"))
    args = ap.parse_args()

    artifacts = Path(args.artifacts_dir)
    y0 = joblib.load(artifacts / "day19_outcome_mu0_pipe.joblib")
    y1 = joblib.load(artifacts / "day19_outcome_mu1_pipe.joblib")

    with open(artifacts / "DAY19_metadata.json","r") as f:
        meta = json.load(f)

    X_cols = meta["X_cols"]
    df = pd.read_csv(args.input)

    # minimal checks
    missing = [c for c in X_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in input: {missing}")

    X = df[X_cols].copy()
    mu0 = y0.predict_proba(X)[:,1]
    mu1 = y1.predict_proba(X)[:,1]
    uplift = mu0 - mu1

    out = df[["appointment_id","person_id"]].copy()
    out["p_no_show_no_sms"] = mu0
    out["p_no_show_sms"] = mu1
    out["uplift"] = uplift

    if args.strategy == "risk":
        out["score"] = out["p_no_show_no_sms"]
    else:
        out["score"] = out["uplift"]

    out = out.sort_values("score", ascending=False).reset_index(drop=True)
    out["rank"] = out.index + 1
    out["send_sms"] = (out["rank"] <= args.budget_k).astype(int)

    out.to_csv(args.output, index=False)

if __name__ == "__main__":
    main()
'''
(SRC / "run_policy.py").write_text(run_policy_py, encoding="utf-8")
print("Wrote:", SRC / "run_policy.py")


Wrote: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-19\src\run_policy.py


### Write Day-19/DAY19.md

In [18]:
md = f"""# Day 19 — Decision + Deployment (SMS targeting)

This day turns model outputs into deployable **policy outputs**. Using the appointments feature table in DuckDB (`{FEATURES_TBL}`) and patient-level splits (`{SPLITS_TBL}`), we trained two outcome models:

- **mu0(x)**: predicted probability of **no-show** if **no SMS** is sent.
- **mu1(x)**: predicted probability of **no-show** if **SMS** is sent.

We define **uplift(x) = mu0(x) − mu1(x)**. Positive uplift means sending an SMS is expected to reduce no-shows.

We then produce two decision policies under a daily budget **K = {K}**:

1. **Risk policy**: send SMS to the top-K highest baseline risk `mu0`.
2. **Uplift policy**: send SMS to the top-K highest uplift.

Outputs saved in `Day-19/reports/` include:

- `DAY19_policy_value_table.csv`
- `DAY19_decisions_test_k{K}.csv`
- `DAY19_target_list_risk_k{K}.csv`
- `DAY19_target_list_uplift_k{K}.csv`

Artifacts saved in `Day-19/artifacts/` include:

- `day19_outcome_mu0_pipe.joblib`, `day19_outcome_mu1_pipe.joblib`
- `day19_propensity_pipe.joblib`
- `DAY19_input_template.csv`
- `DAY19_metadata.json`

DuckDB tables written:

- `policy_runs`
- `policy_recommendations`

Run id: `{RUN_ID}` at `{RUN_TS}`.
"""
(REPORTS / "DAY19.md").write_text(md, encoding="utf-8")
print("Wrote:", REPORTS / "DAY19.md")


Wrote: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-19\reports\DAY19.md


### Close DuckDB cleanly (prevents file-lock hell)

In [19]:
con.close()
print("Closed DuckDB connection.")


Closed DuckDB connection.
